# Stock source assessment

Read-only profiling of a local NASDAQ CSV snapshot. This notebook calls the same profiling function used by the project, without calling its command-line entry point or writing a profile file. Only aggregate counts are displayed; the raw data and per-file manifest remain local.

Open from the repository or project folder, use Python 3 with an `ipykernel` Jupyter environment, and run all cells. Set `STOCK_SOURCE_DIR` if the data is outside the local `datasets/Stocks - Nasdaq` folder. Source files are Git-ignored and must be obtained separately. No outputs are saved in this notebook.

In [1]:
from pathlib import Path
import importlib.util, os, sys
sys.dont_write_bytecode = True

def find_repo():
    for parent in (Path.cwd(), *Path.cwd().parents):
        if (parent / 'projects' / 'stock-market-analytics' / 'pipelines' / 'profile_sources.py').is_file():
            return parent
    raise FileNotFoundError('Open this notebook from the repository or stock project folder.')

repo = find_repo()
source = Path(os.environ['STOCK_SOURCE_DIR']).expanduser() if os.environ.get('STOCK_SOURCE_DIR') else repo / 'datasets' / 'Stocks - Nasdaq'
if not source.is_dir() or not any(source.glob('*.csv')):
    raise FileNotFoundError('Stock CSV files not found; set STOCK_SOURCE_DIR to the local source folder.')
spec = importlib.util.spec_from_file_location('stock_profiler', repo / 'projects' / 'stock-market-analytics' / 'pipelines' / 'profile_sources.py')
profiler = importlib.util.module_from_spec(spec)
spec.loader.exec_module(profiler)
print('Local files found. Profiling can take a few minutes; no source paths or rows will be displayed.')

Local files found. Profiling can take a few minutes; no source paths or rows will be displayed.


## Inventory and schema

The project profiler counts files and records, hashes each source, checks headers, and tracks the combined date range. Date coverage differs by stock and does not establish continuity.

In [2]:
result = profiler.profile(source)
print('Files:', result['file_count'])
print('Total bytes:', result['total_bytes'])
print('Records:', result['counts'].get('rows', 0))
print('Combined date range:', result['date_range']['first'], 'to', result['date_range']['last'])
print('Files ending on latest date:', result['files_ending_on_latest_date'])
print('Files with at least five calendar years of history:', result['files_with_five_year_history_at_latest'])
for header in result['headers']:
    print('Header:', ', '.join(header['columns']), '| files:', header['file_count'])

Files: 499
Total bytes: 118730017
Records: 2740359
Combined date range: 1962-01-02 to 2026-11-16
Files ending on latest date: 498
Files with at least five calendar years of history: 439
Header: ticker, date, open, high, low, close | files: 499


## Quality checks

The profiler checks dates and ticker identity within each file, price parsing, nonpositive prices, and the OHLC relationship. The categories can overlap; do not add them together.

In [3]:
for name in ('invalid_date', 'duplicate_ticker_date', 'ticker_filename_mismatch', 'invalid_price', 'out_of_order_date', 'unexpected_header', 'nonpositive_price', 'ohlc_range_inconsistent', 'ohlc_inconsistent_with_nonpositive_open'):
    print(name + ':', result['counts'].get(name, 0))
print('Nonpositive cells by field:', result['nonpositive_by_field'])
print('Files with nonpositive prices:', result['files_with_nonpositive_prices'])
print('Inconsistent OHLC rows by year:', result['ohlc_inconsistent_by_year'])

invalid_date: 0
duplicate_ticker_date: 0
ticker_filename_mismatch: 0
invalid_price: 0
out_of_order_date: 0
unexpected_header: 0
nonpositive_price: 102324
ohlc_range_inconsistent: 102329
ohlc_inconsistent_with_nonpositive_open: 102324
Nonpositive cells by field: {'open': 102324}
Files with nonpositive prices: 78
Inconsistent OHLC rows by year: {'1962': 504, '1963': 502, '1964': 506, '1965': 504, '1966': 504, '1967': 502, '1968': 452, '1969': 500, '1972': 2, '1973': 1411, '1974': 1714, '1975': 1722, '1976': 1703, '1977': 1719, '1978': 1779, '1979': 2020, '1980': 5733, '1981': 7352, '1982': 7805, '1983': 8412, '1984': 8508, '1985': 7865, '1986': 8827, '1987': 8971, '1988': 8478, '1989': 8687, '1990': 3381, '1991': 1933, '1992': 328, '2006': 1, '2009': 3, '2011': 1}


## Interpretation

These checks do not verify prices against an independent source, establish a trading calendar, or determine dividend and split adjustment treatment. Returns should be described as changes in supplied close prices, not total returns. The profiler identifies duplicate ticker/date pairs within each file; cross-file conflicts require separate checks. See [source assessment](../docs/source-assessment.md) and [profiler implementation](../pipelines/profile_sources.py).